In [35]:
!pip install shap

   ---------------------------------------- 0.0/530.3 kB ? eta -:--:--
   ------------------- -------------------- 262.1/530.3 kB ? eta -:--:--
   ---------------------------------------- 530.3/530.3 kB 2.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   -------------- ------------------------- 1.0/2.8 MB 2.4 MB/s eta 0:00:01
   ---------------------- ----------------- 1.6/2.8 MB 2.3 MB/s eta 0:00:01
   ------------------------- -------------- 1.8/2.8 MB 2.2 MB/s eta 0:00:01
   ----------------------------- ---------- 2.1/2.8 MB 2.1 MB/s eta 0:00:01
   ------------------------------------- -- 2.6/2.8 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 2.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/30.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/30.3 MB ? eta -:--:--
    --------------------------------------


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error
import pickle

# ----- Step 1: Generate Synthetic Data -----

# Seed for reproducibility
np.random.seed(42)
random.seed(42)

# Define sample data lists
countries = [
    "USA", "Canada", "UK", "Germany", "France", "Spain", "Italy", "Australia",
    "Japan", "China", "India", "Brazil", "South Africa", "Russia", "Netherlands"
]
currencies = {
    "USA": "USD", "Canada": "CAD", "UK": "GBP", "Germany": "EUR", "France": "EUR",
    "Spain": "EUR", "Italy": "EUR", "Australia": "AUD", "Japan": "JPY", "China": "CNY",
    "India": "INR", "Brazil": "BRL", "South Africa": "ZAR", "Russia": "RUB", "Netherlands": "EUR"
}
transaction_types = ["Wire", "SWIFT", "ACH", "SEPA", "RTGS"]

def random_date():
    start_date = datetime.now() - timedelta(days=365)
    random_days = np.random.randint(0, 365)
    random_seconds = np.random.randint(0, 86400)
    return start_date + timedelta(days=random_days, seconds=random_seconds)

def assign_jurisdiction_risk(risk_score):
    if risk_score < 40:
        return "Low"
    elif risk_score < 70:
        return "Medium"
    else:
        return "High"

# Number of records
n_records = 10000

# Generate synthetic data
transaction_ids = [f"TX-{100000+i}" for i in range(n_records)]
timestamps = [random_date() for _ in range(n_records)]
sender_countries = [random.choice(countries) for _ in range(n_records)]
receiver_countries = [random.choice(countries) for _ in range(n_records)]
amounts = np.round(np.random.uniform(100, 100000, n_records), 2)
transaction_type_choices = [random.choice(transaction_types) for _ in range(n_records)]
risk_scores = np.round(np.random.uniform(0, 100, n_records), 2)
jurisdiction_risks = [assign_jurisdiction_risk(score) for score in risk_scores]
compliance_flags = [True if score > 70 else False for score in risk_scores]
transaction_currencies = [currencies[sender] for sender in sender_countries]

# Build DataFrame
df = pd.DataFrame({
    "TransactionID": transaction_ids,
    "Timestamp": timestamps,
    "SenderCountry": sender_countries,
    "ReceiverCountry": receiver_countries,
    "Amount": amounts,
    "Currency": transaction_currencies,
    "TransactionType": transaction_type_choices,
    "RiskScore": risk_scores,
    "JurisdictionRisk": jurisdiction_risks,
    "ComplianceFlag": compliance_flags,
    "GeneratedByGenAI": ["Yes"] * n_records
})

# ----- Step 2: Prepare Data for Model Training -----
# For training the model, we'll use the following features:
# - SenderCountry, ReceiverCountry, TransactionType (categorical)
# - Amount (numerical)
# Target variable: RiskScore

# Select features and target
features = df[["SenderCountry", "ReceiverCountry", "TransactionType", "Amount"]]
target = df["RiskScore"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Define the preprocessing for categorical features: one-hot encoding
categorical_features = ["SenderCountry", "ReceiverCountry", "TransactionType"]
numeric_features = ["Amount"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"  # numeric features are left untouched
)

# ----- Step 3: Build and Train the Model Pipeline -----
# We create a pipeline that first preprocesses the data and then trains a Random Forest Regressor.

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train the model
model_pipeline.fit(X_train, y_train)

# ----- Step 4: Evaluate the Model -----
# Predict on the test set
y_pred = model_pipeline.predict(X_test)

# Calculate the root mean squared error (RMSE)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Test RMSE:", rmse)

# ----- Step 5: Save the Trained Model -----
# Save the model to a file for later use (e.g., for predicting new transactions)
with open("risk_score_model.pkl", "wb") as f:
    pickle.dump(model_pipeline, f)

print("Model training complete and saved as 'risk_score_model.pkl'.")

# ----- Step 6: Example of Making a Prediction for a New Transaction -----
# Create a sample new transaction record (you can replace these values with real inputs)
new_transaction = pd.DataFrame({
    "SenderCountry": ["USA"],
    "ReceiverCountry": ["India"],
    "TransactionType": ["SWIFT"],
    "Amount": [25000.00]
})

# Load the model (if needed) and predict
with open("risk_score_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

predicted_risk_score = loaded_model.predict(new_transaction)        
print("Predicted Risk Score for the new transaction:", predicted_risk_score[0])


In [41]:
new_transaction = pd.DataFrame({
    "SenderCountry": ["USA"],
    "ReceiverCountry": ["India"],
    "TransactionType": ["ACH"],
    "Amount": [25000.00]
})

# Load the model (if needed) and predict
with open("risk_score_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

predicted_risk_score = loaded_model.predict(new_transaction)
print("Predicted Risk Score for the new transaction:", predicted_risk_score[0])

Predicted Risk Score for the new transaction: 50.4325


In [4]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error
import pickle

# ----- Step 1: Generate Synthetic Data -----

# Seed for reproducibility
np.random.seed(42)
random.seed(42)

# Define sample data lists
countries = [
    "USA", "Canada", "UK", "Germany", "France", "Spain", "Italy", "Australia",
    "Japan", "China", "India", "Brazil", "South Africa", "Russia", "Netherlands"
]
currencies = {
    "USA": "USD", "Canada": "CAD", "UK": "GBP", "Germany": "EUR", "France": "EUR",
    "Spain": "EUR", "Italy": "EUR", "Australia": "AUD", "Japan": "JPY", "China": "CNY",
    "India": "INR", "Brazil": "BRL", "South Africa": "ZAR", "Russia": "RUB", "Netherlands": "EUR"
}
transaction_types = ["Wire", "SWIFT", "ACH", "SEPA", "RTGS"]

def random_date():
    start_date = datetime.now() - timedelta(days=365)
    random_days = np.random.randint(0, 365)
    random_seconds = np.random.randint(0, 86400)
    return start_date + timedelta(days=random_days, seconds=random_seconds)

def assign_jurisdiction_risk(risk_score):
    if risk_score < 40:
        return "Low"
    elif risk_score < 70:
        return "Medium"
    else:
        return "High"

# Number of records
n_records = 10000

# Generate synthetic data
transaction_ids = [f"TX-{100000+i}" for i in range(n_records)]
timestamps = [random_date() for _ in range(n_records)]
sender_countries = [random.choice(countries) for _ in range(n_records)]
receiver_countries = [random.choice(countries) for _ in range(n_records)]
amounts = np.round(np.random.uniform(100, 100000, n_records), 2)
transaction_type_choices = [random.choice(transaction_types) for _ in range(n_records)]
risk_scores = np.round(np.random.uniform(0, 100, n_records), 2)
jurisdiction_risks = [assign_jurisdiction_risk(score) for score in risk_scores]
compliance_flags = [True if score > 70 else False for score in risk_scores]
transaction_currencies = [currencies[sender] for sender in sender_countries]

# Build DataFrame
df = pd.DataFrame({
    "TransactionID": transaction_ids,
    "Timestamp": timestamps,
    "SenderCountry": sender_countries,
    "ReceiverCountry": receiver_countries,
    "Amount": amounts,
    "Currency": transaction_currencies,
    "TransactionType": transaction_type_choices,
    "RiskScore": risk_scores,
    "JurisdictionRisk": jurisdiction_risks,
    "ComplianceFlag": compliance_flags,
    "GeneratedByGenAI": ["Yes"] * n_records
})

# ----- Step 2: Prepare Data for Model Training -----
# Features: SenderCountry, ReceiverCountry, TransactionType (categorical) and Amount (numeric)
# Target variable: RiskScore

features = df[["SenderCountry", "ReceiverCountry", "TransactionType", "Amount"]]
target = df["RiskScore"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Preprocessing pipelines for both categorical and numeric features
categorical_features = ["SenderCountry", "ReceiverCountry", "TransactionType"]
numeric_features = ["Amount"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

# ----- Step 3: Build the Model Pipeline with Hyperparameter Tuning -----
# We will use GridSearchCV to tune hyperparameters of the RandomForestRegressor.

# Create the pipeline with preprocessor and regressor
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42))
])

# Define parameter grid for GridSearchCV
param_grid = {
    "regressor__n_estimators": [100, 200],
    "regressor__max_depth": [None, 10, 20],
    "regressor__min_samples_split": [2, 5]
}

# Set up GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring="neg_mean_squared_error", n_jobs=-1, verbose=1)

# Train the model using GridSearchCV
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# ----- Step 4: Evaluate the Best Model -----
y_pred = best_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Best Model Test RMSE:", rmse)
print("Best Parameters:", grid_search.best_params_)

# ----- Step 5: Save the Trained Model -----
with open("improved_risk_score_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
print("Improved model training complete and saved as 'improved_risk_score_model.pkl'.")

# ----- Step 6: Example of Making a Prediction for a New Transaction -----
new_transaction = pd.DataFrame({
     "SenderCountry": ["Russia"],        
    "ReceiverCountry": ["South Africa"],
    "TransactionType": ["SWIFT"],       
    "Amount": [95000.00] 
})

with open("improved_risk_score_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

predicted_risk_score = loaded_model.predict(new_transaction)
print("Predicted Risk Score for the new transaction:", predicted_risk_score[0])


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Model Test RMSE: 28.950924775641415
Best Parameters: {'regressor__max_depth': 10, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 200}
Improved model training complete and saved as 'improved_risk_score_model.pkl'.
Predicted Risk Score for the new transaction: 48.23722197122154
